[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kewel-corp/book-vol1/blob/main/notebooks/ch37_code.ipynb)

# 第37章　データ拡張を実装する：2次元と3次元

**『医療診断支援AI開発　第1巻　入門編 ― 動かして、自分で作る（第1巻　入門編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-vol1

## 弾性変形：適用条件と注意点

In [ ]:
import albumentations as A

train_tf = A.Compose([
    # ---- 幾何変換：画像とマスクの両方に、同じ変換を適用する ----
    A.HorizontalFlip(p=0.5),                       # 対象タスクとラベルの整合性を確認して採用する
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,   # 新しい版では A.Affine(...) が推奨
                       rotate_limit=15, p=0.7),
    A.ElasticTransform(alpha=30, sigma=6, p=0.3),  # 弾性変形（生体の形のばらつきを模す）
    # ---- 強度変換：画像だけに適用する（マスクは変わらない）----
    A.RandomBrightnessContrast(0.15, 0.15, p=0.5),
    A.RandomGamma(gamma_limit=(85, 115), p=0.3),   # ガンマ補正（装置ごとの階調差を模す）
    # ノイズ（低線量CTのざらつき）。albumentations 2.x の書式。std_range は最大値に対する比で、
    # (0.01, 0.02) は 1.x の var_limit=(5, 25)（標準偏差2.2〜5.0）にほぼ相当する。
    # 2.x に var_limit を渡すと警告が出るだけで無視され、既定の std_range=(0.2, 0.44)
    # が適用される。これは画像の画素値が0や1に飽和するおそれのある強度なので注意。
    A.GaussNoise(std_range=(0.01, 0.02), p=0.3),
])

out = train_tf(image=img, mask=mask)               # mask= に指定すると画像とマスクに同じ幾何変換を適用する
aug_img, aug_mask = out["image"], out["mask"]

## 37.2　2次元：torchvisionで

In [ ]:
from torchvision.transforms import v2
import torch

train_tf = v2.Compose([
    v2.ToImage(),                                # PILやndarrayをテンソルへ（val_tfと同じ）
    v2.RandomResizedCrop(224, scale=(0.8, 1.0)),
    v2.RandomRotation(15),                       # ±15度
    v2.ColorJitter(brightness=0.1, contrast=0.1),# 変化の幅は±10%
    v2.RandomHorizontalFlip(p=0.5),              # 対象タスクとラベルの整合性を確認して採用する
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406],     # 学習も検証と同じ正規化（省略すると学習時と検証時の入力分布がずれる）
                 std=[0.229, 0.224, 0.225]),
])
val_tf = v2.Compose([                            # 検証は拡張なし
    v2.ToImage(),                                # PILやndarrayをテンソルへ
    v2.Resize((224, 224)),                       # ※Resize(224)だと短辺だけ224になり、形がそろわない
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406],     # 事前学習と同じ正規化（省略すると事前学習時の入力条件と異なる）
                 std=[0.229, 0.224, 0.225]),
])

## 37.3　3次元：MONAIで

In [ ]:
from monai import transforms as T

train_tf3d = T.Compose([
    # RAS基準では 空間軸0=左右, 軸1=前後, 軸2=頭尾。
    # 以下は左右反転を採用すると判断した課題の例。標準的な撮影像と異なる変換も含め、
    # 対象タスクとラベルの整合性を確認し、変換の有無による性能を検証して選ぶ。
    T.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),   # 左右のみ（表32で可否を判断）
    # 90度回転は横断面の向きを変える。標準的な撮影像と異なる向きも生じるため、
    # 対象タスク・ラベルとの整合性を確認し、回転の有無による性能を検証して採用する。
    T.RandRotate90d(keys=["image", "label"], prob=0.3, spatial_axes=(0, 1)),
    T.RandScaleIntensityd(keys="image", factors=0.1, prob=0.3),       # 画像の強度を変化させる
    T.RandGaussianNoised(keys="image", prob=0.2, std=0.02),           # ノイズ
    T.RandAffined(keys=["image", "label"], prob=0.3,                  # アフィン変形（回転・スケール）
                  # rotate_range の単位はラジアン（0.1 rad は約5.7度）。前節の rotate_limit=15 /
                  # RandomRotation(15) は「度」なので、単位を取り違えて 15 と指定すると約859度になり、
                  # エラーが出ず、意図しない回転範囲になる。またスカラーを渡すと空間軸0にしか回転が
                  # 適用されない ― 3軸すべてに適用したいなら rotate_range=(0.1, 0.1, 0.1) と書く。
                  rotate_range=0.1, scale_range=0.1,
                  mode=("bilinear", "nearest")),                       # ラベルは最近傍
    # 局所的な非剛体ワープ（弾性変形）を試すなら T.Rand3DElasticd を使う
])

## 37.4　MixUp：2枚を混ぜる

In [ ]:
import numpy as np, torch

def mixup(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0))
    mixed_x = lam * x + (1 - lam) * x[idx]       # 画像を混ぜる
    return mixed_x, y, y[idx], lam               # 2つのラベルと混合比を返す

# 使い方：損失も、2つのラベルで同じ比率に按分する
mixed_x, ya, yb, lam = mixup(x, y)
out = model(mixed_x)
loss = lam * criterion(out, ya) + (1 - lam) * criterion(out, yb)

## 数字で追う：各変換が画素をどう書き換えるか

In [ ]:
import numpy as np
patch = np.array([[0.0, 0.1, 0.2, 0.3],
                  [0.4, 0.5, 0.6, 0.7],
                  [0.8, 0.9, 1.0, 0.9],
                  [0.7, 0.5, 0.3, 0.1]])
mask  = np.array([[1, 1, 0, 0],
                  [1, 1, 0, 0],
                  [1, 1, 0, 0],
                  [1, 1, 0, 0]])

flip   = patch[:, ::-1]            # 左右反転：列の順を逆に並べ替える
rot90  = np.rot90(patch)          # 90度回転：軸を入れ替え、片方の順序を反転する
bright = np.clip(patch * 1.3, 0, 1)  # 明るさ+30%：値を一律に1.3倍し、1を上限にする
rng    = np.random.default_rng(0)
noisy  = np.clip(patch + rng.normal(0, 0.05, patch.shape), 0, 1)  # ガウスノイズ